# OE1: reproducción autónoma en Google Colab

**Objetivo oficial (literal).** Determinar la relación entre el cumplimiento de los requerimientos funcionales y no funcionales de la arquitectura multiagente y el desempeño del proceso de adaptación de contenido multimodal, mediante indicadores cuantificables de funcionalidad, confiabilidad, eficiencia y calidad de adaptación.

**Qué comprueba este cuaderno.** (a) Recalcula, desde cada petición registrada por el generador de carga, los indicadores por corrida (peticiones, errores, ventana, throughput, P50, media y P95 de la latencia, T_conv medio) de las 120 corridas válidas y los compara con lo guardado por el ejecutor; (b) reejecuta el análisis pre-registrado completo (puntaje de cumplimiento S, requerimientos, confiabilidad, eficiencia, calidad de adaptación, contrastes pareados, correlaciones intra-bloque e interferencia) y lo compara con el resultado local.

**Datos de entrada.** Registros por petición `raw_http/*.csv` y banderas por ciclo `*.oe1.csv` de la campaña, `runs.jsonl`, `intentos.jsonl`, perfiles del conjunto de datos, referencia gold sellada, monitor de recursos y el script de análisis sellado (`analisis_oe1.py`), con los resultados locales para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Pulse «Ejecutar todo» en un entorno nuevo de Colab: los datos se descargan solos desde GitHub (commit fijado) y se verifican por SHA-256; no hay que subir ni buscar archivos.

**Contenido.** 1 descripción · 2 entorno y código (clonado de la etiqueta inmutable) · 3 descarga automática y verificación SHA-256 · 4 validación de integridad y estructura · 5 preprocesamiento y cálculo de indicadores (scripts sellados) · 6 análisis estadístico · 7 comparación con el resultado local y con las cifras de la tesis · 8 tablas · 9 gráficos · 10 pruebas automáticas · 11 resumen. Las diferencias se clasifican como exacta, dentro de tolerancia, explicada por versión, no explicada o no reproducible.

## 1. Entorno y código de análisis
El código vive en el repositorio público (módulos legibles en `reproducibility/etapa26_colab_autonomo/src/repro/`); los scripts de análisis sellados vienen dentro de los paquetes de datos, con su hash verificado.

In [ ]:
import os, sys, platform, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Renato5Lara/multiagent-llm-education.git'
REPO_REF = 'repro-etapa26-v1'            # etiqueta inmutable del codigo; los datos estan fijados por commit en manifests/data_manifest.json
if os.environ.get('REPRO_LOCAL_REPO'):   # solo para pruebas locales antes de publicar la etiqueta
    REPO_DIR = os.environ['REPRO_LOCAL_REPO']
else:
    REPO_DIR = 'repo_repro'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', REPO_REF, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'reproducibility', 'etapa26_colab_autonomo', 'src'))
import repro
from repro import sh
print('Codigo de analisis:', REPO_URL, '@', REPO_REF)


In [ ]:
try:
    import statsmodels
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'statsmodels'], check=True)
    import statsmodels
import statsmodels, numpy, scipy, pandas, matplotlib
ENTORNO = dict(python=platform.python_version(), plataforma=platform.platform(), numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__, matplotlib=matplotlib.__version__, cpu=os.cpu_count(), statsmodels=statsmodels.__version__)
print(ENTORNO)


## 2. Descarga automática y verificación de los paquetes (SHA-256)

In [ ]:
# Descarga automatica desde GitHub (commit fijado) y verificacion SHA-256 contra el manifiesto. No hay que subir nada.
# Si falta un archivo, no se puede descargar o su hash no coincide, se detiene con el archivo, la URL y el hash esperado.
PAQUETES = ['oe1_http_p1']
repro.fetch_packages(PAQUETES, dest='.')


## 3. Verificación interna, cálculo y análisis (celdas del notebook de comprobación publicado, sin cambios salvo la ejecución de scripts)

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('http/hashes_paquete_http_oe1.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); n+=1
    if sha(os.path.join('http',nm))!=h: bad.append(nm)
print('archivos del paquete HTTP verificados contra el hash local:', n, '| distintos:', bad[:5]); assert not bad
base='http/oe1/campana_01'; ok=0; mal=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join(base,nm)
    if os.path.exists(p):
        ok+=1
        if sha(p)!=h: mal.append(nm)
print('oe1: archivos de la campaña presentes y verificados contra su SHA256SUMS.txt:', ok, '| distintos:', mal); assert not mal

In [ ]:
## Recálculo por corrida desde las peticiones registradas

In [ ]:
os.makedirs('http/out', exist_ok=True)
for k in ('oe1',):
    sh(f"python http/recomputo_http.py http/{k}/campana_01 http/out/recomputo_{k}.csv http/out/resumen_{k}.json > http/out/log_{k}.txt 2>&1")
import json
print('Recálculo INDEPENDIENTE, desde cada petición registrada por el generador de carga, de los indicadores por corrida (12 campos) y comparación con lo guardado por el ejecutor:')
for k in ('oe1',):
    r=json.load(open(f'http/out/resumen_{k}.json'))
    mx=max(r['diferencia_abs_max_por_campo'].values()); mr=max(r['diferencia_rel_max_por_campo'].values())
    print(f"{k}: intentos recalculados {r['intentos_con_raw_recalculados']} | corridas válidas finales recalculadas {r['corridas_validas_finales_recalculadas']} | con los 12 campos idénticos {r['corridas_validas_finales_con_todos_los_campos_iguales']} | diferencia absoluta máxima {mx} | relativa máxima {mr}")

In [ ]:
## Análisis pre-registrado de OE1 desde los datos originales

In [ ]:
sh(f"python http/oe1/analisis_oe1.py http/oe1/campana_01 http/oe1/salida http/oe1/gold_sellado.json http/oe1/campana_01_monitor.csv > http/oe1/log_analisis.txt 2>&1")
print(open('http/oe1/log_analisis.txt').read()[-300:])
for f in ('celdas.csv','corridas_indicadores.csv','efectos.csv','resultados_oe1.json'):
    print(f, 'SHA-256 idéntico al resultado local:', sha('http/oe1/salida/'+f)==sha('http/oe1/esperado_local/'+f))

In [ ]:
import pandas as pd
import json, math
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
def comparar_json(ruta_colab, ruta_local, etiqueta):
    A=dict(hoja(json.load(open(ruta_colab,encoding='utf8')))); B=dict(hoja(json.load(open(ruta_local,encoding='utf8'))))
    mismas = set(A)==set(B); mx=0.0; mxrel=0.0; peor=None; n_num=0; n_dist=0; n_nonum=0; dist=[]
    for k in A:
        if k not in B: continue
        va, vb = A[k], B[k]
        if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool) and not isinstance(vb,bool):
            n_num+=1; dif=abs(va-vb); rel=dif/max(abs(vb),1e-300) if dif>0 else 0.0
            if dif>mx: mx=dif; peor=(k,va,vb)
            mxrel=max(mxrel, rel)
            if dif>0: n_dist+=1
            if dif>1e-9 and rel>1e-9: dist.append((k,va,vb,dif,rel))
        else:
            n_nonum+=1
            if va!=vb: dist.append((k,va,vb,None,None))
    print(f'[{etiqueta}] mismas claves: {mismas} | hojas: {len(A)} (numericas {n_num}, no numericas {n_nonum})')
    print(f'   diferencia absoluta maxima: {mx:.3e} | relativa maxima: {mxrel:.3e} | hojas numericas con diferencia > 0: {n_dist} | hojas con diferencia > 1e-9 (abs y rel): {len(dist)}')
    print('   peor hoja:', peor)
    for d_ in dist[:20]: print('   DIFERENCIA', d_)
def comparar_csv(ruta_colab, ruta_local, etiqueta):
    a=pd.read_csv(ruta_colab); b=pd.read_csv(ruta_local)
    ok = list(a.columns)==list(b.columns) and a.shape==b.shape
    mx=0.0; mxrel=0.0; nd=0
    if ok:
        for c in a.columns:
            if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c]) and not pd.api.types.is_bool_dtype(a[c]) and not pd.api.types.is_bool_dtype(b[c]):
                dif=(a[c]-b[c]).abs().fillna(0); mx=max(mx,float(dif.max()))
                rel=(dif/b[c].abs().clip(lower=1e-300)).where(dif>0,0.0); mxrel=max(mxrel,float(rel.max())); nd+=int((dif>0).sum())
            else:
                nd+=int((a[c].astype(object).where(a[c].notna(),'<NA>').astype(str)!=b[c].astype(object).where(b[c].notna(),'<NA>').astype(str)).sum())
    print(f'[{etiqueta}] misma forma y columnas: {ok} {a.shape} | dif abs max: {mx:.3e} | rel max: {mxrel:.3e} | celdas distintas: {nd}')
comparar_json('http/oe1/salida/resultados_oe1.json','http/oe1/esperado_local/resultados_oe1.json','OE1 pre-registrado: resultados_oe1.json')
for f in ('celdas.csv','corridas_indicadores.csv','efectos.csv'):
    comparar_csv('http/oe1/salida/'+f,'http/oe1/esperado_local/'+f,'OE1 pre-registrado: '+f)

In [ ]:
a=json.load(open('http/oe1/salida/resultados_oe1.json')); b=json.load(open('http/oe1/esperado_local/resultados_oe1.json'))
print('registros/válidas/interferidas:', [a['n_registros'],a['n_validas'],a['interferidas']], '|', [b['n_registros'],b['n_validas'],b['interferidas']])
print('variación de S (min, max, niveles):', a['variacion_de_S'], '|', b['variacion_de_S'])
for k in ('p95','thr','tconv','f1','c','err'):
    print('rho intra-bloque con S', k, round(a['relacion_S_respuesta'][k]['rho_intra_bloque'],4), '|', round(b['relacion_S_respuesta'][k]['rho_intra_bloque'],4))
for phi in ('0.0','0.02','0.05'):
    print('I-13 (Wilson, ciclos agrupados) phi', phi, round(a['confiabilidad_I13'][phi]['ciclos_pool']['est'],4), '|', round(b['confiabilidad_I13'][phi]['ciclos_pool']['est'],4))

## 4. Comparación con las cifras de la tesis
Cada cifra se lee del resultado **reproducido** y se compara con la que figura en la tesis (versión CIERRE_FINAL) y en el JSON oficial sellado (`manifests/valores_oficiales.json`).

In [ ]:
from repro import official, validation, tables, figures
from repro.data_loading import sha256_file
import pandas as pd
RUTAS = {'resultados_oe1.json': 'http/oe1/salida/resultados_oe1.json', 'celdas.csv': 'http/oe1/salida/celdas.csv'}
cifras = official.evaluar('OE1', RUTAS)
print(cifras[['id', 'descripcion', 'esperado', 'obtenido', 'dif', 'estado']].to_string(index=False))


## 5. Clasificación de las diferencias frente al resultado local
Exacta · dentro de tolerancia (1e-9) · explicada por versión · no explicada · no reproducible.

In [ ]:
import scipy
REG = validation.Registro(scipy_version=scipy.__version__)
COMPARACIONES = [('OE1 resultados_oe1.json', 'http/oe1/salida/resultados_oe1.json', 'http/oe1/esperado_local/resultados_oe1.json'), ('OE1 celdas.csv', 'http/oe1/salida/celdas.csv', 'http/oe1/esperado_local/celdas.csv'), ('OE1 corridas_indicadores.csv', 'http/oe1/salida/corridas_indicadores.csv', 'http/oe1/esperado_local/corridas_indicadores.csv'), ('OE1 efectos.csv', 'http/oe1/salida/efectos.csv', 'http/oe1/esperado_local/efectos.csv')]
for _e, _a, _b in COMPARACIONES:
    REG.comparar(_e, _a, _b)


## 6. Tablas

In [ ]:
OUT = 'outputs'
TABLAS = {'celdas': 'http/oe1/salida/celdas.csv', 'efectos': 'http/oe1/salida/efectos.csv', 'corridas_indicadores': 'http/oe1/salida/corridas_indicadores.csv'}
tablas_generadas = tables.exportar_tablas('OE1', TABLAS, os.path.join(OUT, 'tables'))
print('Tablas generadas:')
for _t in tablas_generadas:
    print('  ', _t)


## 7. Gráficos
Se generan desde la salida del análisis ejecutado arriba (no desde valores copiados). Se exportan PNG (300 ppi), SVG y el CSV de la serie dibujada (`figure_data/`).

In [ ]:
METAS = official.cargar_valores()['metas_de_diseno']
fig = figures.fig_oe1(RUTAS['resultados_oe1.json'], os.path.join(OUT, 'figures', 'OE1'), metas=METAS)
ref = figures.fig_oe1('http/oe1/esperado_local/resultados_oe1.json', os.path.join(OUT, 'figures_ref_desde_resultado_local', 'OE1'), metas=METAS)
REG.comparar('Serie de la Figura 1 reproducida frente a la serie dibujada desde el resultado oficial local', fig['serie_csv'], ref['serie_csv'])
_t = official.cargar_valores()['figuras_tesis']['fig_oe1']
IDENTICA = sha256_file(fig['png']) == _t['sha256_png_tesis']
print('Figura 1: PNG identico bit a bit al de la tesis (pagina %s del indice): %s' % (_t['pagina_indice'], IDENTICA))
if not IDENTICA:
    print('  Esperable fuera del entorno de la tesis (matplotlib 3.10.3 y fuente Arial): la imagen puede diferir en pixeles; la comprobacion valida es la de la serie dibujada.')
try:
    from IPython.display import Image, display
    display(Image(fig['png']))
except Exception:
    pass


## 8. Pruebas automáticas de consistencia

In [ ]:
fallos = []
if cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE']).any():
    fallos.append('cifras oficiales: ' + str(cifras[cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE'])][['id', 'estado']].values.tolist()))
peor, cuenta = REG.veredicto()
if peor in ('NO_EXPLICADA', 'NO_REPRODUCIBLE'):
    fallos.append('comparacion con el resultado local: ' + peor)
if not (os.path.getsize(fig['png']) > 0 and os.path.getsize(fig['serie_csv']) > 0):
    fallos.append('figura vacia')
assert not fallos, fallos
print('PRUEBAS AUTOMATICAS DE CONSISTENCIA: todas superadas | cifras oficiales comprobadas:', len(cifras), '| comparaciones:', cuenta)


## 9. Resumen final de la reproducción

In [ ]:
import shutil
resumen = tables.guardar_resumen('OE1', os.path.join(OUT, 'reports'), REG.tabla(), cifras, ENTORNO, extra=dict(veredicto=peor, conteo=cuenta, paquetes=PAQUETES))
print('RESUMEN DE REPRODUCCION OE1')
print(REG.tabla()[['comparacion', 'categoria', 'dif_abs_max']].to_string(index=False))
print('Veredicto (peor categoria):', peor, cuenta)
print('Cifras de la tesis comprobadas: %d (%d coinciden exactamente, %d dentro del redondeo)' % (len(cifras), (cifras.estado == 'COINCIDE').sum(), (cifras.estado == 'COINCIDE_REDONDEO').sum()))
print('LIMITACIONES: las 120 corridas tuvieron interferencia ambiental; la confiabilidad cae con la perdida de mensajes; RNF03 no se cumple (F1_adapt 0,8044 < 0,85); campana no controlada.')
print('Alcance: se reproduce el analisis, no la adquisicion de datos; no se repite ninguna campana.')
_z = shutil.make_archive('resultados_OE1', 'zip', OUT)
print('Salidas empaquetadas en', os.path.basename(_z))
if IN_COLAB:
    from google.colab import files
    files.download(_z)
